In [1]:
import numpy as np
import pandas as pd
from itertools import product

def concentration_score(file_R0='R0.csv', 
                        file_sigma='sigma.csv',
                        true_R0=None,
                        true_sigma=None,
                        selected_indices=None):

    data_R0 = pd.read_csv(file_R0, header=None).values.ravel()
    data_sigma = pd.read_csv(file_sigma, header=None).values.ravel()
    
    # print(f"Loaded {len(data_R0)} {len(data_sigma)} samples")
    selected_R0 = data_R0[selected_indices]
    selected_sigma = data_sigma[selected_indices]
    
    d_R0    = ((selected_R0 - true_R0)    ** 2).mean()
    d_sigma = ((selected_sigma - true_sigma) ** 2).mean()
    return d_R0 + d_sigma

def compute_distance(filepath: str,
    standard_point: tuple,
    weights: tuple = None,
    metric: str = "euclidean",  # "euclidean", "manhattan", "chebyshev", "minkowski", "cosine"
    p: float = 3,               # only used when metric="minkowski"
    ):
                     
     # 1. Load
    df = pd.read_csv(filepath, header=None, names=["A", "B", "C", "D"])

    # 2. Min-Max normalization
    col_min = df.min()
    col_max = df.max()
    df_norm = (df - col_min) / (col_max - col_min)

    # 3. Normalize the standard point on the same scale
    standard = np.array(standard_point)
    standard_norm = (standard - col_min.values) / (col_max.values - col_min.values)

    # 4. Resolve weights (normalize so they sum to 1)
    if weights is not None:
        w = np.array(weights, dtype=float)
        if len(w) != 4:
            raise ValueError("weights must have exactly 4 values (wA, wB, wC, wD).")
        if np.any(w < 0):
            raise ValueError("All weights must be non-negative.")
        w = w / w.sum()          # normalize to sum = 1
    else:
        w = np.array([0.25, 0.25, 0.25, 0.25])   # equal weights

    # 5. Weighted different distance functions: Euclidean, manhattan, chebyshev, minkowski, cosine
    diff = (df_norm[["A", "B", "C", "D"]] - standard_norm).values
    if metric == "euclidean":
        dist = np.sqrt((w * diff ** 2).sum(axis=1))

    elif metric == "manhattan":
        dist = (w * np.abs(diff)).sum(axis=1)

    elif metric == "chebyshev":
        dist = (w * np.abs(diff)).max(axis=1)

    elif metric == "minkowski":
        dist = ((w * np.abs(diff) ** p).sum(axis=1)) ** (1 / p)

    elif metric == "cosine":
        dot     = (w * df_norm[["A", "B", "C", "D"]].values * standard_norm).sum(axis=1)
        norm_x  = np.sqrt((w * df_norm[["A", "B", "C", "D"]].values ** 2).sum(axis=1))
        norm_x0 = np.sqrt((w * standard_norm ** 2).sum())
        dist    = 1 - dot / (norm_x * norm_x0)

    else:
        raise ValueError(f"Unknown metric '{metric}'. Choose from: euclidean, manhattan, chebyshev, minkowski, cosine.")

    df_norm["distance"] = dist

    return df_norm["distance"]

In [2]:
##### setting: R0=3.5, sigma=0.7

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.5, 0.7   # ← your true values
percentile = 0.05
standard_point=(5.20e+02, 5.67e+04, 4.48e-01, 2.64e+01)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
5912  (0.9, 0.1, 0.9, 0.9)  6.894948  8000
5911  (0.9, 0.1, 0.9, 0.8)  6.911452  8000
5183  (0.8, 0.1, 0.9, 0.9)  6.924333  8000
5910  (0.9, 0.1, 0.9, 0.7)  6.928886  8000
3725  (0.6, 0.1, 0.9, 0.9)  6.937542  8000
2996  (0.5, 0.1, 0.9, 0.9)  6.937542  8000
5180  (0.8, 0.1, 0.9, 0.6)  6.942436  8000
5906  (0.9, 0.1, 0.9, 0.3)  6.942624  8000
1538  (0.3, 0.1, 0.9, 0.9)  6.944782  8000
2267  (0.4, 0.1, 0.9, 0.9)  6.944782  8000
2266  (0.4, 0.1, 0.9, 0.8)  6.944782  8000
5182  (0.8, 0.1, 0.9, 0.8)  6.945425  8000
5181  (0.8, 0.1, 0.9, 0.7)  6.945425  8000
5908  (0.9, 0.1, 0.9, 0.5)  6.945425  8000
5909  (0.9, 0.1, 0.9, 0.6)  6.945425  8000
4454  (0.7, 0.1, 0.9, 0.9)  6.951198  8000
4453  (0.7, 0.1, 0.9, 0.8)  6.951198  8000
4452  (0.7, 0.1, 0.9, 0.7)  6.951198  8000
3724  (0.6, 0.1, 0.9, 0.8)  6.951198  8000
5907  (0.9, 0.1, 0.9, 0.4)  6.951653  8000
4451  (0.7, 0.1, 0.9, 0.6)  6.953320  8000
2995  (0.5, 0.1, 0.9, 0.8)  6.95

In [3]:
##### setting: R0=2.0, sigma=0.7

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.0, 0.7   # ← your true values
percentile = 0.05
standard_point=(31.7826087, 20.19891204, -0.47262852,  8.60776849)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R01p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R01p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R01p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(50))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
576   (0.1, 0.8, 0.2, 0.1)  0.544786  8000
657   (0.1, 0.9, 0.2, 0.1)  0.547160  8000
495   (0.1, 0.7, 0.2, 0.1)  0.553863  8000
648   (0.1, 0.9, 0.1, 0.1)  0.559278  8000
1296  (0.2, 0.8, 0.1, 0.1)  0.561656  8000
666   (0.1, 0.9, 0.3, 0.1)  0.562769  8000
1377  (0.2, 0.9, 0.1, 0.1)  0.563783  8000
2106  (0.3, 0.9, 0.1, 0.1)  0.563957  8000
486   (0.1, 0.7, 0.1, 0.1)  0.564584  8000
568   (0.1, 0.8, 0.1, 0.2)  0.565025  8000
1386  (0.2, 0.9, 0.2, 0.1)  0.565902  8000
2025  (0.3, 0.8, 0.1, 0.1)  0.566407  8000
649   (0.1, 0.9, 0.1, 0.2)  0.566972  8000
405   (0.1, 0.6, 0.1, 0.1)  0.567933  8000
567   (0.1, 0.8, 0.1, 0.1)  0.569186  8000
658   (0.1, 0.9, 0.2, 0.2)  0.571181  8000
1215  (0.2, 0.7, 0.1, 0.1)  0.571467  8000
675   (0.1, 0.9, 0.4, 0.1)  0.572454  8000
651   (0.1, 0.9, 0.1, 0.4)  0.575247  8000
585   (0.1, 0.8, 0.3, 0.1)  0.575867  8000
650   (0.1, 0.9, 0.1, 0.3)  0.576193  8000
2107  (0.3, 0.9, 0.1, 0.2)  0.57